### Packrat Parsing for Statements

The following cells are from the course notes.

Given a parsing expression grammar `P`, a recursive descent parser with backtracking reading global variable `src` with the input is constructed as follows. For each production `p`, the parser `pr(p)` that starts parsing at input position `k` is:

| `p`             | `pr(p)`                      |
|:----------------|:-----------------------------|
| `B ← E`         | `procedure B(k: integer) → integer \| Fail` <br> `pr(E)` <br> `return k` |

The rules for constructing `pr(E)` for recognizing `E` starting at position `k` in `src` are:

| `E`            | `pr(E)`                             |
|:---------------|:------------------------------------|
| `'ε'`          | `skip` |
| `'a'`          | `if prefix('a', src[k:]) then k := k + len(a) else k := Fail` |
| `.`            | `if k < len(src) then k := k + 1 else k := Fail` |
| `B`            | `k ← B(k)`                               |
| `(E₁)`         | `pr(E₁)`                            |
| `E₁?`          | `var r  := k` <br> `pr(E₁)` <br> `if k = Fail then k := r` |
| `E₁*`          | `var r` <br> `while k ≠ Fail do` <br>  `r := k ; pr(E₁)` <br> `k := r` |
| `E₁+`          | `pr(E₁)` <br> `var r := k` <br> `while k ≠ Fail do` <br>  `r := k ; pr(E₁)` <br> `k := r` |
| `&E`           | `var r := k` <br> `pr(E₁)` <br> `if k ≠ Fail then k := r` |
| `!E`           | `var r := k` <br> `pr(E₁)` <br> `if k = Fail then k := r else k := Fail` |
| `E₁ E₂  …`     | `pr(E₁)` <br>`if k ≠ Fail then pr(E₂)` <br>`…`       |
| `E₁ / E₂ /  …` | `var r := k` <br> `pr(E₁)` <br> `if k = Fail then` <br>  `k := r ; pr(E₂)` <br>  `…`  |

The procedure of the start symbol must be called to recognize a sentence in the language.

Consider PEG `P₀`:

    S ← A / B
    A ← x A / y
    B ← x B / z

The resulting recursive descent parser with backtracking is expressed as a Python class with the source as a field; failing parsing procedures return `None`:

In [ ]:
class P0Backtrack:
    src: str

    def literal(self, k: int, a: str) -> int | None:
        if self.src.startswith(a, k): return k + len(a) # else return None

    def S(self, k: int) -> int | None: # S ← A / B
        r = k; k = self.A(k)
        if k == None: k = self.B(r)
        return k

    def A(self, k: int) -> int | None: # A ← x A / y
        r = k; k = self.literal(k, 'x')
        if k != None: k = self.A(k)
        if k == None: k = self.literal(r, 'y')
        return k

    def B(self, k: int) -> int | None: # B ← x B / z
        r = k; k = self.literal(k, 'x')
        if k != None: k = self.B(k)
        if k == None: k = self.literal(r, 'z')
        return k

    def parse(self, s: str) -> bool:
        self.src = s
        return self.S(0) == len(s)

In [ ]:
p0 = P0Backtrack()
assert not p0.parse('') and not p0.parse('x') and p0.parse('y') and p0.parse('z')
assert p0.parse('xy') and p0.parse('xz') and not p0.parse('xx') and p0.parse('xxz')

---

Consider the following PEG for assignment statements and procedure calls:

    statement ← 
        ident selector ':=' ident /
        ident (',' ident)* ':=' ident (',' ident)* /
        ident (',' ident)* '←' ident '(' ident ')'
    selector ← ('[' ident ']' / '.' ident)*
    ident  ← 'a' / ... / 'z'

Implement the parser as a backtracking parser, following the rules from the course notes. You may use the commented version of method `literal`.

In [ ]:
class StatementBacktrack:
    src: str

    def literal(self, k: int, a: str):
        if self.src.startswith(a, k): return k + len(a)
        # if self.src.startswith(a, k): print('match', a, 'at', k); return k + len(a)
        # else: print('fail', a, 'at', k); return None

    # YOUR CODE HERE
    raise NotImplementedError()
    
    def parse(self, s: str):
        self.src = s; return self.statement(0) == len(s)

In [ ]:
pb = StatementBacktrack()

Here are some tests:

In [ ]:
assert pb.parse('a:=b')
assert pb.parse('a.b.c:=d')
assert pb.parse('a[b].c[e]:=f')
assert pb.parse('a,b,c:=d,e,f')
assert pb.parse('a,b,c←d(e)')
assert not pb.parse('a')
assert not pb.parse('a:b')
assert not pb.parse('a.b.(c)')
assert not pb.parse('a..b:=c')
assert not pb.parse('a.b:=7')

Now write a memoizing parser:

In [ ]:
class StatementMemoizing(StatementBacktrack):
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
pm = StatementMemoizing()

Here are the same tests:

In [ ]:
assert pb.parse('a:=b')
assert pb.parse('a.b.c:=d')
assert pb.parse('a[b].c[e]:=f')
assert pb.parse('a,b,c:=d,e,f')
assert pb.parse('a,b,c←d(e)')
assert not pb.parse('a')
assert not pb.parse('a:b')
assert not pb.parse('a.b.(c)')
assert not pb.parse('a..b:=c')
assert not pb.parse('a.b:=7')